# NumPy and pandas

**Family:** Fundamentals | **Level:** Beginner | **Runs on:** CPU, under a minute

Arrays, tables, missing values and grouping on a real dataset.

## Learning objectives

- Index, slice and broadcast arrays
- Select, filter and group a DataFrame
- Find and handle missing values
- Avoid common copy and view mistakes

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ml_lab.data import load_dataset
from ml_lab.utils import set_seed
from ml_lab import visualization as viz
SEED = 42
set_seed(SEED)
viz.apply_style()

## 1. Arrays and broadcasting

Broadcasting lets arrays of different shapes combine without loops.

In [2]:
A = np.arange(12).reshape(3, 4)
print(A, "\nshape", A.shape)
print("column means:", A.mean(axis=0), "| row sums:", A.sum(axis=1))
print("centre columns by broadcasting:\n", A - A.mean(axis=0))

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]] 
shape (3, 4)
column means: [4. 5. 6. 7.] | row sums: [ 6 22 38]
centre columns by broadcasting:
 [[-4. -4. -4. -4.]
 [ 0.  0.  0.  0.]
 [ 4.  4.  4.  4.]]


## 2. A real table: Auto MPG

The dataset registry loads real data with a licence note.

In [3]:
bundle = load_dataset("auto_mpg")
df = bundle.frame
display(df.head())
print(df.dtypes.to_dict())
print("missing values per column:", df.isna().sum().to_dict())

,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,mpg
0,8.0,307.0,130.0,3504.0,12.0,70.0,1.0,18.0
1,8.0,350.0,165.0,3693.0,11.5,70.0,1.0,15.0
2,8.0,318.0,150.0,3436.0,11.0,70.0,1.0,18.0
3,8.0,304.0,150.0,3433.0,12.0,70.0,1.0,16.0
4,8.0,302.0,140.0,3449.0,10.5,70.0,1.0,17.0


{'cylinders': dtype('float64'), 'displacement': dtype('float64'), 'horsepower': dtype('float64'), 'weight': dtype('float64'), 'acceleration': dtype('float64'), 'model_year': dtype('float64'), 'origin': dtype('float64'), 'mpg': dtype('float64')}
missing values per column: {'cylinders': 0, 'displacement': 0, 'horsepower': 0, 'weight': 0, 'acceleration': 0, 'model_year': 0, 'origin': 0, 'mpg': 0}


## 3. Select, filter, group

In [4]:
heavy = df[df["weight"] > df["weight"].median()]
print("heavy cars:", len(heavy), "| mean mpg heavy:", round(heavy["mpg"].mean(), 1), "| mean mpg light:", round(df[df["weight"] <= df["weight"].median()]["mpg"].mean(), 1))
display(df.groupby("origin")[["mpg", "horsepower", "weight"]].agg(["mean", "std"]).round(1))

heavy cars: 196 | mean mpg heavy: 17.6 | mean mpg light: 29.2


mpg      horsepower        weight       
        mean  std       mean   std    mean    std
origin                                           
1.0     20.0  6.4      119.0  39.9  3372.5  795.3
2.0     27.6  6.6       80.6  20.2  2433.5  491.8
3.0     30.5  6.1       79.8  17.8  2221.2  320.5

## 4. Missing values

We poke holes into a copy to practise. Real imputation inside a pipeline is in the preprocessing module.

In [5]:
holey = df.copy()
rng = np.random.default_rng(SEED)
holey.loc[rng.choice(holey.index, 30, replace=False), "horsepower"] = np.nan
print("missing:", int(holey["horsepower"].isna().sum()))
print("mean imputation changes the std from", round(df["horsepower"].std(), 1), "to", round(holey["horsepower"].fillna(holey["horsepower"].mean()).std(), 1))

missing: 30
mean imputation changes the std from 38.5 to 37.6


## 5. Views and copies

Changing a slice may or may not change the original. Use `.copy()` when you mean a copy.

In [6]:
v = A[:, :2]
v[0, 0] = 99
print("original changed through a view:", A[0, 0] == 99)
c = A[:, :2].copy(); c[0, 0] = -1
print("original changed through a copy:", A[0, 0] == -1)

original changed through a view: True
original changed through a copy: False


## Exercises

1. Compute the correlation of `mpg` with every other column and sort it.
2. Show that median imputation changes the standard deviation less than a constant fill of 0.

## Summary

- Broadcasting replaces loops.
- Group and aggregate before modelling.
- Imputation changes distributions.

**Next:** Data visualization.

**Data note.** Real datasets come from `ml_lab.data` with licence notes in `datasets/catalog`. Synthetic numbers appear only where a demonstration needs controlled data, and are labelled.